In [5]:
import cv2
import numpy as np
from rknnlite.api import RKNNLite


class Yolo26Seg:
    def __init__(self, model_path: str, input_size: int = 640, conf_thresh: float = 0.25, iou_thresh: float = 0.45):
        self.conf_thresh = conf_thresh
        self.iou_thresh = iou_thresh
        self.input_size = input_size
        self.num_classes = 80
        self.proto_channel = 32

        self.rknn = RKNNLite()
        if self.rknn.load_rknn(model_path) != 0:
            raise RuntimeError("RKNN 모델 로드 실패")
        if self.rknn.init_runtime() != 0:
            raise RuntimeError("RKNN Runtime 초기화 실패")

    def letterbox(self, im, color=(0, 0, 0)):
        shape = im.shape[:2]
        ratio = min(self.input_size / shape[0], self.input_size / shape[1])
        new_unpad = int(round(shape[1] * ratio)), int(round(shape[0] * ratio))

        if shape[::-1] != new_unpad:
            im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)

        bottom = self.input_size - new_unpad[1]
        right = self.input_size - new_unpad[0]
        im = cv2.copyMakeBorder(im, 0, bottom, 0, right, cv2.BORDER_CONSTANT, value=color)

        return im, ratio, (0, 0, right, bottom)

    def _preprocess(self, image: np.ndarray):
        h, w = image.shape[:2]
        img_letterbox, ratio, pad = self.letterbox(image)
        
        img_rgb = cv2.cvtColor(img_letterbox, cv2.COLOR_BGR2RGB)
        img_in = np.expand_dims(img_rgb, axis=0)  # RKNN Lite NHWC 형태 입력
        
        return img_in, (h, w), ratio, pad

    def _postprocess(self, outputs: list, orig_shape: tuple, ratio: float, pad: tuple):
        orig_h, orig_w = orig_shape
        x_pad, y_pad = pad[2], pad[3]
        
        # 1. Output6: Proto Mask
        proto = outputs[6]
        if proto.ndim == 4:
            proto = proto[0]  # (32, 160, 160)
        proto_h, proto_w = proto.shape[1], proto.shape[2]

        head_outputs = [
            (outputs[0], outputs[1]),  # Stride 8  (80x80)
            (outputs[2], outputs[3]),  # Stride 16 (40x40)
            (outputs[4], outputs[5])   # Stride 32 (20x20)
        ]

        filter_boxes = []
        obj_probs = []
        class_ids = []
        filter_segments = []

        for det_out, seg_out in head_outputs:
            # det_out shape: (1, 84, grid_h, grid_w)
            # seg_out shape: (1, 32, grid_h, grid_w)
            det = det_out[0]
            seg = seg_out[0]
            
            grid_h, grid_w = det.shape[1], det.shape[2]
            stride = self.input_size // grid_h

            # C++의 process_fp32 내부 2중 Loop 디코딩을 정확하게 구현[cite: 1]
            for i in range(grid_h):
                for j in range(grid_w):
                    # 클래스 별 Score 추출: offset (4+cls)*grid_w*grid_h + i*grid_w + j[cite: 1]
                    cls_scores = det[4:4 + self.num_classes, i, j]
                    max_cls_id = int(np.argmax(cls_scores))
                    box_conf = float(cls_scores[max_cls_id])

                    if box_conf >= self.conf_thresh:
                        # BBox loc 디코딩[cite: 1]
                        loc = det[:4, i, j]
                        x1 = (j + 0.5 - loc[0]) * stride
                        y1 = (i + 0.5 - loc[1]) * stride
                        w_ = (loc[0] + loc[2]) * stride
                        h_ = (loc[1] + loc[3]) * stride

                        filter_boxes.append([x1, y1, w_, h_])
                        obj_probs.append(box_conf)
                        class_ids.append(max_cls_id)
                        
                        # Seg 계수 추출 (32 차원)[cite: 1]
                        filter_segments.append(seg[:, i, j])

        if not filter_boxes:
            return [], [], [], None

        # NMS 수행
        boxes_for_nms = [[b[0], b[1], b[2], b[3]] for b in filter_boxes]
        indices = cv2.dnn.NMSBoxes(boxes_for_nms, obj_probs, self.conf_thresh, self.iou_thresh)
        if len(indices) == 0:
            return [], [], [], None

        indices = indices.flatten()
        boxes_num = len(indices)

        final_boxes = []
        crop_boxes = []
        final_scores = []
        final_class_ids = []
        selected_segments = []

        for idx in indices:
            bx, by, bw, bh = filter_boxes[idx]
            x1, y1 = bx, by
            x2, y2 = bx + bw, by + bh

            crop_boxes.append([x1, y1, x2, y2])
            
            # C++ box_reverse 역변환[cite: 1]
            real_x1 = int(np.clip(x1 / ratio, 0, orig_w))
            real_y1 = int(np.clip(y1 / ratio, 0, orig_h))
            real_x2 = int(np.clip(x2 / ratio, 0, orig_w))
            real_y2 = int(np.clip(y2 / ratio, 0, orig_h))

            final_boxes.append([real_x1, real_y1, real_x2, real_y2])
            final_scores.append(obj_probs[idx])
            final_class_ids.append(class_ids[idx])
            selected_segments.append(filter_segments[idx])

        # Matmul 연산: (N, 32) @ (32, 160*160) -> (N, 160, 160)[cite: 1]
        selected_segments = np.array(selected_segments)  # (N, 32)
        proto_flat = proto.reshape(self.proto_channel, -1)  # (32, 25600)
        matmul_out = np.dot(selected_segments, proto_flat).reshape(boxes_num, proto_h, proto_w)

        # 160x160 -> 640x640 확대[cite: 1]
        seg_mask = np.zeros((boxes_num, self.input_size, self.input_size), dtype=np.float32)
        for b in range(boxes_num):
            seg_mask[b] = cv2.resize(matmul_out[b], (self.input_size, self.input_size), interpolation=cv2.INTER_LINEAR)

        # C++ crop_mask_fp 정확히 반영[cite: 1]
        all_mask_in_one = np.zeros((self.input_size, self.input_size), dtype=np.uint8)
        for b in range(boxes_num):
            cx1, cy1, cx2, cy2 = [int(v) for v in crop_boxes[b]]
            cx1, cy1 = max(0, cx1), max(0, cy1)
            cx2, cy2 = min(self.input_size, cx2), min(self.input_size, cy2)

            if cx2 <= cx1 or cy2 <= cy1:
                continue

            sub_seg = seg_mask[b, cy1:cy2, cx1:cx2]
            sub_mask = all_mask_in_one[cy1:cy2, cx1:cx2]
            
            mask_cond = (sub_mask == 0) & (sub_seg > 0)
            sub_mask[mask_cond] = final_class_ids[b] + 1

        # C++ seg_reverse (Letterbox 패딩 잘라내기 후 원본 스케일링)[cite: 1]
        cropped_h = self.input_size - y_pad
        cropped_w = self.input_size - x_pad
        cropped_seg = all_mask_in_one[:cropped_h, :cropped_w]

        real_seg_mask = cv2.resize(cropped_seg, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

        return final_boxes, final_scores, final_class_ids, real_seg_mask

    def infer(self, image: np.ndarray):
        img_in, orig_shape, ratio, pad = self._preprocess(image)
        outputs = self.rknn.inference(inputs=[img_in])
        return self._postprocess(outputs, orig_shape, ratio, pad)

    def draw_results(self, image: np.ndarray, boxes, scores, class_ids, real_seg_mask):
        res_img = image.copy()

        # 마스크 시각화
        if real_seg_mask is not None and np.any(real_seg_mask > 0):
            colored_mask = np.zeros_like(res_img, dtype=np.uint8)
            unique_classes = np.unique(real_seg_mask)

            for cid_plus_1 in unique_classes:
                if cid_plus_1 == 0:
                    continue
                np.random.seed(int(cid_plus_1))
                color = np.random.randint(0, 255, (3,), dtype=np.uint8)
                colored_mask[real_seg_mask == cid_plus_1] = color

            mask_pos = real_seg_mask > 0
            overlay = cv2.addWeighted(res_img, 0.5, colored_mask, 0.5, 0)
            res_img[mask_pos] = overlay[mask_pos]

        # BBox 시각화
        for box, score, cid in zip(boxes, scores, class_ids):
            np.random.seed(cid + 1)
            color = np.random.randint(0, 255, (3,), dtype=np.uint8).tolist()

            x1, y1, x2, y2 = box
            cv2.rectangle(res_img, (x1, y1), (x2, y2), color, 2)
            label = f"Cls {cid}: {score:.2f}"
            cv2.putText(res_img, label, (x1, max(y1 - 10, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        return res_img

    def release(self):
        self.rknn.release()

RK3588_RKNN_MODEL = './models/seg/yolo26/yolov26n-seg-RK3588_640.rknn'

IMG_PATH = './images/bus.jpg'
OBJ_THRESH = 0.25
IMG_SIZE = 640

# --- 사용 예시 ---
if __name__ == "__main__":
    detector = Yolo26Seg(model_path=RK3588_RKNN_MODEL, input_size=IMG_SIZE)
    frame = cv2.imread(IMG_PATH)

    if frame is not None:
        boxes, scores, class_ids, real_seg_mask = detector.infer(frame)
        result_img = detector.draw_results(frame, boxes, scores, class_ids, real_seg_mask)
        cv2.imwrite('result.jpg', result_img)
        print(f"검출 완료: {len(boxes)}개 객체")

    detector.release()

W Query dynamic range failed. Ret code: RKNN_ERR_MODEL_INVALID. (If it is a static shape RKNN model, please ignore the above warning message.)


I RKNN: [13:44:23.032] RKNN Runtime Information, librknnrt version: 2.4.0 (b458df3b4a@2026-01-17T10:53:35)
I RKNN: [13:44:23.032] RKNN Driver Information, version: 0.9.8
I RKNN: [13:44:23.033] RKNN Model Information, version: 6, toolkit version: 2.3.2(compiler version: 2.3.2 (@2025-04-03T08:26:16)), target: RKNPU v2, target platform: rk3588, framework name: ONNX, framework layout: NCHW, model inference type: static_shape
W RKNN: [13:44:23.060] query RKNN_QUERY_INPUT_DYNAMIC_RANGE error, rknn model is static shape type, please export rknn with dynamic_shapes
검출 완료: 4개 객체


In [12]:
import cv2
import numpy as np
from rknnlite.api import RKNNLite


def hex2rgb(h):  # rgb order (PIL)
    return tuple(int(h[1 + i:1 + i + 2], 16) for i in (0, 2, 4))


class Colors:
    # Ultralytics color palette https://ultralytics.com/
    def __init__(self):
        hexs = ('FF3838', 'FF9D97', 'FF701F', 'FFB21D', 'CFD231', '48F90A', '92CC17', '3DDB86', '1A9334', '00D4BB',
                '2C99A8', '00C2FF', '344593', '6473FF', '0018EC', '8438FF', '520085', 'CB38FF', 'FF95C8', 'FF37C7')
        self.palette = [hex2rgb(f'#{c}') for c in hexs]
        self.n = len(self.palette)

    def __call__(self, i, bgr=False):
        c = self.palette[int(i) % self.n]
        return (c[2], c[1], c[0]) if bgr else c


colors = Colors()


class Yolo26Seg:
    def __init__(self, model_path: str, input_size: int = 640, conf_thresh: float = 0.25, iou_thresh: float = 0.45):
        self.conf_thresh = conf_thresh
        self.iou_thresh = iou_thresh
        self.input_size = input_size
        self.num_classes = 80
        self.proto_channel = 32

        self.rknn = RKNNLite()
        if self.rknn.load_rknn(model_path) != 0:
            raise RuntimeError("RKNN 모델 로드 실패")
        if self.rknn.init_runtime() != 0:
            raise RuntimeError("RKNN Runtime 초기화 실패")

    def letterbox(self, im, color=(0, 0, 0)):
        shape = im.shape[:2]
        ratio = min(self.input_size / shape[0], self.input_size / shape[1])
        new_unpad = int(round(shape[1] * ratio)), int(round(shape[0] * ratio))

        if shape[::-1] != new_unpad:
            im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)

        bottom = self.input_size - new_unpad[1]
        right = self.input_size - new_unpad[0]
        im = cv2.copyMakeBorder(im, 0, bottom, 0, right, cv2.BORDER_CONSTANT, value=color)

        return im, ratio

    def _preprocess(self, image: np.ndarray):
        img_letterbox, ratio = self.letterbox(image)
        img_rgb = cv2.cvtColor(img_letterbox, cv2.COLOR_BGR2RGB)
        img_in = np.expand_dims(img_rgb, axis=0)
        
        return img_in, ratio

    def _postprocess(self, outputs: list, ratio: float, orig_shape: tuple = None) -> list[dict]:
        proto = outputs[6]
        if proto.ndim == 4:
            proto = proto[0]

        if proto.shape[-1] == self.proto_channel:
            proto = np.transpose(proto, (2, 0, 1))
        
        proto_h, proto_w = proto.shape[1], proto.shape[2]

        head_outputs = [
            (outputs[0], outputs[1]),  # Stride 8  (80x80)
            (outputs[2], outputs[3]),  # Stride 16 (40x40)
            (outputs[4], outputs[5])   # Stride 32 (20x20)
        ]

        filter_boxes = []
        obj_probs = []
        class_ids = []
        filter_segments = []

        for det_out, seg_out in head_outputs:
            det = det_out[0]
            seg = seg_out[0]

            if det.shape[-1] != det.shape[0] and det.ndim == 3:
                if det.shape[0] != 84 and det.shape[-1] == 84:
                    det = np.transpose(det, (2, 0, 1))
                if seg.shape[0] != 32 and seg.shape[-1] == 32:
                    seg = np.transpose(seg, (2, 0, 1))

            grid_h, grid_w = det.shape[1], det.shape[2]
            stride = self.input_size // grid_h

            for i in range(grid_h):
                for j in range(grid_w):
                    cls_scores = det[4:4 + self.num_classes, i, j]
                    max_cls_id = int(np.argmax(cls_scores))
                    box_conf = float(cls_scores[max_cls_id])

                    if box_conf >= self.conf_thresh:
                        loc = det[:4, i, j]
                        x1 = (j + 0.5 - loc[0]) * stride
                        y1 = (i + 0.5 - loc[1]) * stride
                        w_ = (loc[0] + loc[2]) * stride
                        h_ = (loc[1] + loc[3]) * stride

                        filter_boxes.append([x1, y1, w_, h_])
                        obj_probs.append(box_conf)
                        class_ids.append(max_cls_id)
                        filter_segments.append(seg[:, i, j])

        if not filter_boxes:
            return []

        boxes_for_nms = [[b[0], b[1], b[2], b[3]] for b in filter_boxes]
        indices = cv2.dnn.NMSBoxes(boxes_for_nms, obj_probs, self.conf_thresh, self.iou_thresh)
        if len(indices) == 0:
            return []

        indices = indices.flatten()
        boxes_num = len(indices)

        selected_segments = []
        crop_boxes = []
        nms_boxes = []
        nms_scores = []
        nms_class_ids = []

        for idx in indices:
            bx, by, bw, bh = filter_boxes[idx]
            x1, y1 = bx, by
            x2, y2 = bx + bw, by + bh

            crop_boxes.append([x1, y1, x2, y2])

            real_x1 = int(x1 / ratio)
            real_y1 = int(y1 / ratio)
            real_x2 = int(x2 / ratio)
            real_y2 = int(y2 / ratio)

            nms_boxes.append([real_x1, real_y1, real_x2, real_y2])
            nms_scores.append(obj_probs[idx])
            nms_class_ids.append(class_ids[idx])
            selected_segments.append(filter_segments[idx])

        selected_segments = np.array(selected_segments)
        proto_flat = proto.reshape(self.proto_channel, -1)
        matmul_out = np.dot(selected_segments, proto_flat).reshape(boxes_num, proto_h, proto_w)

        if orig_shape is not None:
            orig_h, orig_w = orig_shape
            cropped_w = int(round(orig_w * ratio))
            cropped_h = int(round(orig_h * ratio))
        else:
            max_x2 = max([b[2] for b in crop_boxes]) if crop_boxes else self.input_size
            max_y2 = max([b[3] for b in crop_boxes]) if crop_boxes else self.input_size
            cropped_w = int(min(self.input_size, max_x2))
            cropped_h = int(min(self.input_size, max_y2))
            orig_w = int(cropped_w / ratio)
            orig_h = int(cropped_h / ratio)

        results = []
        for b in range(boxes_num):
            mask_full = cv2.resize(matmul_out[b], (self.input_size, self.input_size), interpolation=cv2.INTER_LINEAR)

            cx1, cy1, cx2, cy2 = [int(v) for v in crop_boxes[b]]
            cx1, cy1 = max(0, cx1), max(0, cy1)
            cx2, cy2 = min(self.input_size, cx2), min(self.input_size, cy2)

            cropped_inst_mask = np.zeros((self.input_size, self.input_size), dtype=bool)
            if cx2 > cx1 and cy2 > cy1:
                cropped_inst_mask[cy1:cy2, cx1:cx2] = mask_full[cy1:cy2, cx1:cx2] > 0

            valid_crop_mask = cropped_inst_mask[:cropped_h, :cropped_w].astype(np.uint8)
            real_mask = cv2.resize(valid_crop_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST).astype(bool)

            real_box = [
                int(np.clip(nms_boxes[b][0], 0, orig_w)),
                int(np.clip(nms_boxes[b][1], 0, orig_h)),
                int(np.clip(nms_boxes[b][2], 0, orig_w)),
                int(np.clip(nms_boxes[b][3], 0, orig_h))
            ]

            results.append({
                'box': real_box,
                'score': nms_scores[b],
                'class_id': nms_class_ids[b],
                'mask': real_mask
            })

        return results

    def infer(self, image: np.ndarray) -> list[dict]:
        img_in, ratio = self._preprocess(image)
        outputs = self.rknn.inference(inputs=[img_in])
        return self._postprocess(outputs, ratio, orig_shape=image.shape[:2])

    def draw_results(self, image: np.ndarray, results: list[dict]):
        res_img = image.copy()
        img_h, img_w = res_img.shape[:2]

        for res in results:
            box = res['box']
            score = res['score']
            cid = res['class_id']
            mask = res['mask']

            # Colors 팔레트에서 BGR 색상 가져오기
            color = colors(cid, bgr=True)

            if mask is not None and np.any(mask):
                if mask.shape[0] != img_h or mask.shape[1] != img_w:
                    mask = cv2.resize(mask.astype(np.uint8), (img_w, img_h), interpolation=cv2.INTER_NEAREST).astype(bool)

                colored_mask = np.zeros_like(res_img, dtype=np.uint8)
                colored_mask[mask] = color
                overlay = cv2.addWeighted(res_img, 0.5, colored_mask, 0.5, 0)
                res_img[mask] = overlay[mask]

            x1, y1, x2, y2 = box
            cv2.rectangle(res_img, (x1, y1), (x2, y2), color, 2)
            label = f"Cls {cid}: {score:.2f}"
            cv2.putText(res_img, label, (x1, max(y1 - 10, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        return res_img

    def release(self):
        self.rknn.release()

RK3588_RKNN_MODEL = './models/seg/yolo26/yolov26n-seg-RK3588_640_i8.rknn'

IMG_PATH = './images/bus.jpg'
OBJ_THRESH = 0.25
IMG_SIZE = 640

# --- 사용 예시 ---
if __name__ == "__main__":
    detector = Yolo26Seg(model_path=RK3588_RKNN_MODEL, input_size=IMG_SIZE)
    frame = cv2.imread(IMG_PATH)

    if frame is not None:
        results = detector.infer(frame)
        result_img = detector.draw_results(frame, results)
        cv2.imwrite('result_clean.jpg', result_img)

    detector.release()

W Query dynamic range failed. Ret code: RKNN_ERR_MODEL_INVALID. (If it is a static shape RKNN model, please ignore the above warning message.)


I RKNN: [14:13:55.790] RKNN Runtime Information, librknnrt version: 2.4.0 (b458df3b4a@2026-01-17T10:53:35)
I RKNN: [14:13:55.790] RKNN Driver Information, version: 0.9.8
I RKNN: [14:13:55.791] RKNN Model Information, version: 6, toolkit version: 2.3.2(compiler version: 2.3.2 (@2025-04-03T08:26:16)), target: RKNPU v2, target platform: rk3588, framework name: ONNX, framework layout: NCHW, model inference type: static_shape
W RKNN: [14:13:55.804] query RKNN_QUERY_INPUT_DYNAMIC_RANGE error, rknn model is static shape type, please export rknn with dynamic_shapes
